# Lab 02: i primi passi con llama cli

Lab: `corso/laboratori/lab-02-01-primo-modello, lab-02-02-campionamento, lab-02-03-chat-template`

Esegui le celle in ordine (Runtime > Esegui tutto). Scarica fino a 1 GB e richiede alcuni minuti.


## 1. Preparazione

Scarica i binari ufficiali di llama.cpp (build `b11046`, quella del corso) e il materiale del lab.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # thread = core disponibili (oltre i core fisici si rallenta)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Scarica il modello di prova (785 MiB, licenza Apache 2.0) nella cache di Hugging Face.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. Prima domanda (lab 02-01)

`-st` risponde ed esce, `-rea off` spegne il ragionamento, `-t` imposta i thread.

In [ ]:
%%bash
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "Quanto fa 17 per 3? Solo il numero." -n 30 -rea off -t $T < /dev/null 2>&1 | tail -8

Con il ragionamento acceso e pochi token il modello resta dentro `[Start thinking]` senza rispondere; `--reasoning-budget 25` tronca il pensiero e fa arrivare la risposta.

In [ ]:
%%bash
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "Quanto fa 17 per 3?" -n 80 -t $T --reasoning-budget 25 < /dev/null 2>&1 | tail -8

### Prompt di sistema

Stessa domanda, ora con un contesto.

In [ ]:
%%bash
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -rea off -t $T -n 60 \
  -sys "Sei un esperto di modelli linguistici (LLM). Rispondi in italiano, in una frase." \
  -p "Cos'è un token?" < /dev/null 2>&1 | sed -n '/^> /,/Prompt:/p'

## 3. Campionamento (lab 02-02)

Il seed fisso rende la prova ripetibile; `--temp 0` e `--top-k 1` scelgono sempre il token più probabile.

In [ ]:
%%bash
P="Scrivi una frase su un gatto."
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "$P" -n 25 -rea off -t $T --temp 0 < /dev/null 2>&1 | sed -n '/^> /,/Prompt:/p'
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "$P" -n 25 -rea off -t $T -s 1 --temp 1.8 --top-k 0 --top-p 1 --min-p 0.1 < /dev/null 2>&1 | sed -n '/^> /,/Prompt:/p'

## 4. Chat template (lab 02-03)

Il log verbose mostra il formato con cui `llama` impagina i messaggi.

In [ ]:
%%bash
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "Ciao" -n 1 -rea off -t $T -v < /dev/null 2>&1 | grep -A6 "example_format"